# Reactome-only embedding-program and pathway analysis

This companion analysis keeps every upstream result fixed:

- the fine-tuned scGPT checkpoint and 2,439-gene analysis axis;
- the embedding-neighbour graph and 393 Louvain programs;
- observed and model conditional gene effects;
- observed and model gene-program effects and their program-effect FDR values.

Only the annotation library changes. Program over-representation
analysis (ORA) and all four preranked GSEA runs are recalculated
using Reactome alone. This is essential because both ORA
multiple-testing correction and GSEA's pooled-null FDR depend on
the complete family of pathways supplied to the analysis.

The analysis is intentionally stored separately from the original
Hallmark + Reactome outputs. It does not rerun checkpoint inference,
graph construction, Louvain tuning, or program scoring.

## 1. Configuration and imports

In [ ]:
from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import textwrap
import warnings

import gseapy as gp
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import scipy
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.stats import hypergeom, pearsonr, spearmanr
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

ANALYSIS_DIR = Path.cwd().resolve()
OUTPUT_DIR = ANALYSIS_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR = Path("/Users/zhehaowang/PycharmProjects/scgpt_test")
EMBEDDING_DIR = PROJECT_DIR / "Embedding_GRN_save"
SOURCE_OUTPUT_DIR = EMBEDDING_DIR / "perturbation_pathway_outputs"
REACTOME_GMT = (
    EMBEDDING_DIR
    / "resources"
    / "msigdb_2026_1"
    / "c2.cp.reactome.v2026.1.Hs.symbols.gmt"
)

INPUT_FILES = {
    "membership": SOURCE_OUTPUT_DIR / "embedding_gene_program_membership.csv",
    "program_effects": SOURCE_OUTPUT_DIR / "gene_program_effects_observed_vs_model.csv",
    "gene_ranks": SOURCE_OUTPUT_DIR / "gene_effect_rankings.csv",
    "combined_gsea": SOURCE_OUTPUT_DIR / "pathway_gsea_all.csv",
}
for name, path in {**INPUT_FILES, "reactome_gmt": REACTOME_GMT}.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

if OUTPUT_DIR.resolve() == SOURCE_OUTPUT_DIR.resolve():
    raise RuntimeError("Reactome-only outputs must not overwrite the original analysis.")

MIN_GENE_SET_SIZE = 10
MAX_GENE_SET_SIZE = 500
GSEA_PERMUTATIONS = 500
RANDOM_SEED = 42
N_PATHWAYS_PER_TIMEPOINT = 10
REDUNDANCY_JACCARD_THRESHOLD = 0.50

print("Analysis folder:", ANALYSIS_DIR)
print("Output folder:", OUTPUT_DIR)
print("Source outputs:", SOURCE_OUTPUT_DIR)

## 2. Helper functions

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return adjusted
    p = values[finite]
    order = np.argsort(p, kind="mergesort")
    ranked = p[order]
    m = len(ranked)
    q_ranked = ranked * m / np.arange(1, m + 1)
    q_ranked = np.minimum.accumulate(q_ranked[::-1])[::-1]
    q_ranked = np.clip(q_ranked, 0, 1)
    q = np.empty_like(q_ranked)
    q[order] = q_ranked
    adjusted[finite] = q
    return adjusted


def read_reactome_gmt(path):
    raw = OrderedDict()
    with Path(path).open() as handle:
        for line in handle:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 3:
                continue
            raw[fields[0]] = {gene for gene in fields[2:] if gene}
    return raw


def readable_reactome_name(term, width=None):
    label = str(term).split("::", 1)[-1]
    label = label.removeprefix("REACTOME_").replace("_", " ").title()
    replacements = {
        "Dna": "DNA",
        "Rna": "RNA",
        "Mrna": "mRNA",
        "Trna": "tRNA",
        "Ifn": "IFN",
        "Mhc": "MHC",
        "G2 M": "G2/M",
        "G1 S": "G1/S",
        "P53": "p53",
        "Nf Kb": "NF-kB",
        "Igf": "IGF",
        "Igfbps": "IGFBPs",
    }
    for source, target in replacements.items():
        label = label.replace(source, target)
    return textwrap.fill(label, width=width) if width else label


def save_figure(fig, stem, dpi=300):
    stem = Path(stem)
    fig.savefig(stem.with_suffix(".png"), dpi=dpi, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def as_bool(series):
    if series.dtype == bool:
        return series.fillna(False)
    return (
        series.fillna(False)
        .astype(str)
        .str.lower()
        .isin({"true", "1", "yes"})
    )


def safe_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3 or np.nanstd(x[valid]) == 0 or np.nanstd(y[valid]) == 0:
        return np.nan, np.nan
    result = pearsonr(x[valid], y[valid])
    return float(result.statistic), float(result.pvalue)


def safe_spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3:
        return np.nan, np.nan
    result = spearmanr(x[valid], y[valid])
    return float(result.statistic), float(result.pvalue)


def select_representative_pathways(frame, gene_sets, n=10, jaccard_threshold=0.50):
    # Observed-only selection with sign balance and display-only
    # redundancy control.
    ranked = frame.copy()
    ranked["abs_observed_nes"] = ranked["NES_observed"].abs()
    ranked = ranked.sort_values(
        ["FDR q-val_observed", "abs_observed_nes", "Term"],
        ascending=[True, False, True],
    )

    positive = ranked[ranked["NES_observed"] > 0]
    negative = ranked[ranked["NES_observed"] < 0]
    target_positive = int(np.ceil(n / 2))
    target_negative = int(np.floor(n / 2))

    def greedy(candidates, target, selected_terms):
        selected = []
        for _, row in candidates.iterrows():
            genes = gene_sets[row["Term"]]
            redundant = False
            for chosen_term in [
                *selected_terms,
                *[item["Term"] for item in selected],
            ]:
                chosen = gene_sets[chosen_term]
                union = genes | chosen
                jaccard = len(genes & chosen) / len(union) if union else 0.0
                if jaccard >= jaccard_threshold:
                    redundant = True
                    break
            if not redundant:
                selected.append(row)
            if len(selected) >= target:
                break
        return selected

    chosen = greedy(positive, target_positive, [])
    chosen_terms = [item["Term"] for item in chosen]
    chosen.extend(greedy(negative, target_negative, chosen_terms))

    if len(chosen) < n:
        chosen_terms = [item["Term"] for item in chosen]
        for _, row in ranked.iterrows():
            if row["Term"] in chosen_terms:
                continue
            genes = gene_sets[row["Term"]]
            too_similar = False
            for term in chosen_terms:
                other = gene_sets[term]
                union = genes | other
                jaccard = len(genes & other) / len(union) if union else 0.0
                if jaccard >= jaccard_threshold:
                    too_similar = True
                    break
            if not too_similar:
                chosen.append(row.copy())
                chosen_terms.append(row["Term"])
            if len(chosen) >= n:
                break

    selected = (
        pd.DataFrame(chosen).reset_index(drop=True)
        if chosen
        else ranked.head(0).copy()
    )
    selected["selection_basis"] = (
        "observed GSEA q, then absolute observed NES; "
        f"display-only gene-set Jaccard < {jaccard_threshold:.2f}"
    )
    return selected

## 3. Load the fixed upstream results and define the Reactome universe

The gene axis and Louvain membership are not reconstructed here.
The Reactome GMT is intersected with the same 2,439-gene universe,
and only sets containing 10--500 eligible genes are tested.

In [ ]:
membership = pd.read_csv(INPUT_FILES["membership"])
program_effects_source = pd.read_csv(INPUT_FILES["program_effects"])
gene_ranks = pd.read_csv(INPUT_FILES["gene_ranks"])
combined_gsea = pd.read_csv(INPUT_FILES["combined_gsea"])

if "eligible_embedding_gene" in membership.columns:
    membership = membership[as_bool(membership["eligible_embedding_gene"])].copy()
membership["human_symbol"] = membership["human_symbol"].astype(str)
membership["program_id"] = membership["program_id"].astype(str)
membership["module_id"] = membership["module_id"].astype(str)

universe = set(membership["human_symbol"])
rank_universe = set(gene_ranks["human_symbol"].astype(str))
if universe != rank_universe:
    raise AssertionError(
        f"Membership and rank universes differ: {len(universe)} versus {len(rank_universe)}"
    )
if len(universe) != 2439:
    raise AssertionError(f"Expected 2,439 genes, found {len(universe)}")
if membership["program_id"].nunique() != 393:
    raise AssertionError(
        f"Expected 393 programs, found {membership['program_id'].nunique()}"
    )

raw_reactome = read_reactome_gmt(REACTOME_GMT)
reactome_inventory_rows = []
reactome_sets = OrderedDict()
for pathway, source_genes in raw_reactome.items():
    eligible_genes = source_genes & universe
    eligible = MIN_GENE_SET_SIZE <= len(eligible_genes) <= MAX_GENE_SET_SIZE
    term = f"REACTOME::{pathway}"
    reactome_inventory_rows.append(
        {
            "Term": term,
            "pathway": pathway,
            "source_size": len(source_genes),
            "eligible_size": len(eligible_genes),
            "eligible_for_analysis": eligible,
            "eligible_genes": ";".join(sorted(eligible_genes)),
        }
    )
    if eligible:
        reactome_sets[term] = eligible_genes

inventory = pd.DataFrame(reactome_inventory_rows)
inventory["eligible_member_hash"] = inventory["eligible_genes"].map(
    lambda value: hashlib.sha1(value.encode("utf-8")).hexdigest()[:12]
    if value
    else ""
)
inventory["eligible_equivalence_class_size"] = inventory.groupby(
    "eligible_member_hash"
)["Term"].transform("size")
inventory.to_csv(OUTPUT_DIR / "reactome_gene_set_inventory.csv", index=False)

if len(reactome_sets) != 319:
    raise AssertionError(f"Expected 319 eligible Reactome pathways, found {len(reactome_sets)}")

print(f"Fixed gene universe: {len(universe):,}")
print(f"Fixed Louvain programs: {membership['program_id'].nunique():,}")
print(f"Raw Reactome pathways: {len(raw_reactome):,}")
print(f"Eligible Reactome pathways: {len(reactome_sets):,}")
display(
    inventory.groupby("eligible_for_analysis", as_index=False)
    .size()
    .rename(columns={"size": "n_pathways"})
)

## 4. Reactome-only program over-representation analysis

Each fixed Louvain program is tested against each of the 319
eligible Reactome pathways using the upper-tail hypergeometric
probability. The within-program FDR is Benjamini--Hochberg adjusted
across 319 Reactome pathways separately for each program. The
Reactome-global FDR is adjusted across all
\(393\times319=125{,}367\) program--pathway tests.

A provisional annotation requires at least three overlapping genes
and within-program FDR below 0.05. Global FDR is retained and is the
stricter study-wide assessment.

In [ ]:
M = len(universe)
ora_rows = []
for (module_id, program_id), frame in membership.groupby(
    ["module_id", "program_id"], sort=False
):
    symbols = set(frame["human_symbol"])
    program_size = len(symbols)
    for term, pathway_genes in reactome_sets.items():
        overlap = symbols & pathway_genes
        ora_rows.append(
            {
                "module_id": module_id,
                "program_id": program_id,
                "database": "REACTOME",
                "Term": term,
                "pathway": term.split("::", 1)[1],
                "program_size": program_size,
                "pathway_size_in_universe": len(pathway_genes),
                "overlap_size": len(overlap),
                "overlap_genes": ";".join(sorted(overlap)),
                "p_value": float(
                    hypergeom.sf(
                        len(overlap) - 1,
                        M,
                        len(pathway_genes),
                        program_size,
                    )
                ),
            }
        )

reactome_program_ora = pd.DataFrame(ora_rows)
reactome_program_ora["reactome_global_fdr"] = bh_adjust(
    reactome_program_ora["p_value"]
)
reactome_program_ora["reactome_program_fdr"] = (
    reactome_program_ora.groupby("module_id", sort=False)["p_value"]
    .transform(bh_adjust)
)
reactome_program_ora = reactome_program_ora.sort_values(
    ["reactome_global_fdr", "reactome_program_fdr", "p_value", "overlap_size"],
    ascending=[True, True, True, False],
).reset_index(drop=True)

if len(reactome_program_ora) != 393 * 319:
    raise AssertionError(
        f"Expected 125,367 ORA tests, found {len(reactome_program_ora):,}"
    )
if set(reactome_program_ora["database"]) != {"REACTOME"}:
    raise AssertionError("A non-Reactome pathway entered the ORA.")

annotation_rows = []
for (module_id, program_id), frame in reactome_program_ora.groupby(
    ["module_id", "program_id"], sort=False
):
    ranked = frame.sort_values(
        ["reactome_program_fdr", "p_value", "overlap_size"],
        ascending=[True, True, False],
    )
    provisional = ranked[
        ranked["reactome_program_fdr"].lt(0.05)
        & ranked["overlap_size"].ge(3)
    ]
    best = provisional.iloc[0] if not provisional.empty else ranked.iloc[0]
    is_provisional = not provisional.empty
    annotation_rows.append(
        {
            "module_id": module_id,
            "program_id": program_id,
            "reactome_annotation": (
                readable_reactome_name(best["pathway"])
                if is_provisional
                else "Unannotated"
            ),
            "reactome_annotation_is_provisional": is_provisional,
            "reactome_annotation_term": best["Term"] if is_provisional else "",
            "reactome_annotation_program_fdr": (
                float(best["reactome_program_fdr"]) if is_provisional else np.nan
            ),
            "reactome_annotation_global_fdr": (
                float(best["reactome_global_fdr"]) if is_provisional else np.nan
            ),
            "reactome_annotation_overlap_size": (
                int(best["overlap_size"]) if is_provisional else 0
            ),
            "reactome_annotation_overlap_genes": (
                best["overlap_genes"] if is_provisional else ""
            ),
            "best_reactome_term_regardless_of_fdr": best["Term"],
            "best_reactome_p_value": float(best["p_value"]),
        }
    )

reactome_program_annotations = pd.DataFrame(annotation_rows)
reactome_program_ora.to_csv(
    OUTPUT_DIR / "reactome_program_ora_all.csv", index=False
)
reactome_program_annotations.to_csv(
    OUTPUT_DIR / "reactome_program_annotations.csv", index=False
)

provisional_hits = reactome_program_ora[
    reactome_program_ora["reactome_program_fdr"].lt(0.05)
    & reactome_program_ora["overlap_size"].ge(3)
].copy()
provisional_hits.to_csv(
    OUTPUT_DIR / "reactome_program_ora_provisional_hits.csv", index=False
)

ora_summary = pd.DataFrame(
    [
        {
            "n_tests": len(reactome_program_ora),
            "n_provisional_program_pathway_pairs": len(provisional_hits),
            "n_provisionally_annotated_programs": provisional_hits[
                "program_id"
            ].nunique(),
            "n_global_fdr_lt_0_05": int(
                reactome_program_ora["reactome_global_fdr"].lt(0.05).sum()
            ),
            "minimum_global_fdr": float(
                reactome_program_ora["reactome_global_fdr"].min()
            ),
        }
    ]
)
ora_summary.to_csv(OUTPUT_DIR / "reactome_program_ora_summary.csv", index=False)
display(ora_summary)
display(
    provisional_hits[
        [
            "program_id",
            "pathway",
            "overlap_size",
            "overlap_genes",
            "p_value",
            "reactome_program_fdr",
            "reactome_global_fdr",
        ]
    ].sort_values(["reactome_program_fdr", "p_value"])
)

### Reactome-only program-ORA dot plot

In [ ]:
if provisional_hits.empty:
    print("No provisional Reactome program annotations met the plotting rule.")
else:
    dot = provisional_hits.copy()
    dot["label"] = (
        dot["program_id"]
        + " — "
        + dot["pathway"].map(lambda x: readable_reactome_name(x, width=48))
    )
    dot["minus_log10_program_fdr"] = -np.log10(
        dot["reactome_program_fdr"].clip(lower=np.nextafter(0, 1))
    )
    dot = dot.sort_values(
        ["reactome_program_fdr", "overlap_size"],
        ascending=[False, True],
    )
    fig_height = max(5.0, 0.48 * len(dot) + 2.2)
    fig, ax = plt.subplots(figsize=(12.5, fig_height))
    scatter = ax.scatter(
        dot["minus_log10_program_fdr"],
        np.arange(len(dot)),
        s=42 + 28 * dot["overlap_size"],
        c=dot["reactome_global_fdr"],
        cmap="viridis_r",
        norm=Normalize(0, max(0.25, float(dot["reactome_global_fdr"].max()))),
        edgecolor="#202020",
        linewidth=0.5,
    )
    ax.set_yticks(np.arange(len(dot)), dot["label"])
    ax.axvline(-np.log10(0.05), color="#555555", linestyle="--", linewidth=1)
    ax.set_xlabel(r"$-\log_{10}$(Reactome within-program ORA FDR)")
    ax.set_ylabel("")
    ax.set_title(
        "Reactome-only provisional annotations of fixed Louvain programs\n"
        "point size = overlapping genes; colour = Reactome-global ORA FDR\n"
        "no pair passed global FDR < 0.05 (minimum = 0.205)"
    )
    ax.grid(axis="y", visible=False)
    cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
    cbar.set_label("Reactome-global ORA FDR")
    handles = [
        plt.scatter([], [], s=42 + 28 * size, color="#8a8a8a", edgecolor="#202020",
                    linewidth=0.5, label=f"{size} overlapping genes")
        for size in sorted(dot["overlap_size"].unique())
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.005),
        ncol=min(4, len(handles)),
        frameon=False,
    )
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    save_figure(fig, OUTPUT_DIR / "reactome_program_ora_dotplot")

## 5. Reactome-labelled all-program response atlas

The four response columns and hollow program-effect FDR circles are
identical to the original atlas. Only the provisional annotation
labels and orange annotation strip are recalculated from
Reactome-only ORA. Ward clustering of the standardized four-column
response profiles is used solely to order rows for display.

In [ ]:
old_annotation_columns = [
    "annotation",
    "annotation_is_program_fdr_significant",
    "annotation_database",
    "annotation_term",
    "annotation_program_fdr",
    "annotation_global_fdr",
    "annotation_overlap_size",
    "annotation_fdr_scope",
]
program_effects = program_effects_source.drop(
    columns=[c for c in old_annotation_columns if c in program_effects_source.columns]
).merge(
    reactome_program_annotations,
    on=["module_id", "program_id"],
    how="left",
    validate="many_to_one",
)

program_ids = sorted(program_effects["program_id"].unique())
response_values = pd.DataFrame(index=program_ids)
response_fdr = pd.DataFrame(index=program_ids)
response_mapping = (
    ("IR_24h", "observed_delta", "observed_welch_fdr", "Observed 24 h"),
    (
        "IR_24h",
        "predicted_counterfactual_delta",
        "predicted_paired_input_fdr",
        "Model 24 h",
    ),
    ("IR_1week", "observed_delta", "observed_welch_fdr", "Observed 1 week"),
    (
        "IR_1week",
        "predicted_counterfactual_delta",
        "predicted_paired_input_fdr",
        "Model 1 week",
    ),
)
for condition, value_column, fdr_column, label in response_mapping:
    frame = program_effects[
        program_effects["condition"].eq(condition)
    ].set_index("program_id")
    response_values[label] = frame[value_column].reindex(program_ids).astype(float)
    response_fdr[label] = frame[fdr_column].reindex(program_ids).astype(float)

metadata_columns = [
    "program_id",
    "module_id",
    "n_features",
    "n_human_symbols",
    "central_genes",
    "median_seed_jaccard",
    "min_seed_jaccard",
    "reactome_annotation",
    "reactome_annotation_is_provisional",
    "reactome_annotation_program_fdr",
    "reactome_annotation_global_fdr",
]
metadata = (
    program_effects[metadata_columns]
    .drop_duplicates("program_id")
    .set_index("program_id")
    .reindex(response_values.index)
)
metadata["reactome_annotation_is_provisional"] = as_bool(
    metadata["reactome_annotation_is_provisional"]
)

matrix = response_values.to_numpy(dtype=float)
column_sd = np.nanstd(matrix, axis=0, ddof=1)
scaled = (matrix - np.nanmean(matrix, axis=0)) / np.where(
    column_sd > 0, column_sd, 1
)
row_order = leaves_list(linkage(scaled, method="ward", metric="euclidean"))
ordered_values = response_values.iloc[row_order]
ordered_fdr = response_fdr.reindex(ordered_values.index)
ordered_metadata = metadata.reindex(ordered_values.index)

atlas_export = (
    ordered_metadata.join(ordered_values)
    .join(ordered_fdr.add_suffix(" FDR"))
)
atlas_export.insert(0, "display_order", np.arange(1, len(atlas_export) + 1))
atlas_export.to_csv(
    OUTPUT_DIR / "reactome_all_program_response_atlas_data.csv"
)

effect_limit = max(
    float(np.nanmax(np.abs(ordered_values.to_numpy(dtype=float)))), 1e-9
)
sizes = ordered_metadata["n_features"].astype(float).to_numpy()
stability = ordered_metadata["median_seed_jaccard"].astype(float).to_numpy()
annotated = ordered_metadata[
    "reactome_annotation_is_provisional"
].astype(bool).to_numpy()

fig = plt.figure(figsize=(13.4, 15.0))
grid = fig.add_gridspec(
    1,
    4,
    width_ratios=(7.2, 0.48, 0.48, 0.34),
    left=0.22,
    right=0.96,
    bottom=0.12,
    top=0.91,
    wspace=0.08,
)
ax = fig.add_subplot(grid[0, 0])
image = ax.imshow(
    ordered_values.to_numpy(dtype=float),
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    norm=Normalize(-effect_limit, effect_limit),
)
significant_y, significant_x = np.where(
    ordered_fdr.to_numpy(dtype=float) < 0.05
)
ax.scatter(
    significant_x,
    significant_y,
    s=5.5,
    facecolors="none",
    edgecolors="#171717",
    linewidths=0.35,
)
ax.set_xticks(np.arange(len(ordered_values.columns)), ordered_values.columns)
ax.tick_params(
    axis="x", top=True, bottom=False, labeltop=True, labelbottom=False
)

label_positions = np.flatnonzero(annotated).astype(float)
displayed_positions = label_positions.copy()
if len(displayed_positions):
    minimum_gap = 8.0
    for index in range(1, len(displayed_positions)):
        displayed_positions[index] = max(
            displayed_positions[index],
            displayed_positions[index - 1] + minimum_gap,
        )
    overflow = displayed_positions[-1] - (len(ordered_values) - 1)
    if overflow > 0:
        displayed_positions -= overflow
        for index in range(len(displayed_positions) - 2, -1, -1):
            displayed_positions[index] = min(
                displayed_positions[index],
                displayed_positions[index + 1] - minimum_gap,
            )

ax.set_yticks([])
y_transform = ax.get_yaxis_transform()
for true_position, displayed_position in zip(
    label_positions, displayed_positions
):
    program_id = ordered_values.index[int(true_position)]
    annotation = ordered_metadata.iloc[int(true_position)][
        "reactome_annotation"
    ]
    ax.plot(
        [-0.006, -0.024],
        [true_position, displayed_position],
        transform=y_transform,
        color="#777777",
        linewidth=0.45,
        clip_on=False,
    )
    ax.text(
        -0.030,
        displayed_position,
        f"{program_id} — {annotation}",
        transform=y_transform,
        ha="right",
        va="center",
        fontsize=7.2,
        clip_on=False,
    )
ax.set_ylabel(
    "All 393 fixed Louvain embedding programs\n"
    "(only provisional Reactome labels shown)"
)
ax.set_title(
    "Reactome-labelled program-response atlas",
    loc="left",
    fontweight="bold",
)
ax.grid(False)

side_axes = [fig.add_subplot(grid[0, index]) for index in range(1, 4)]
side_axes[0].imshow(
    sizes[:, None],
    aspect="auto",
    interpolation="nearest",
    cmap="Greys",
    norm=Normalize(float(np.nanmin(sizes)), float(np.nanmax(sizes))),
)
side_axes[1].imshow(
    stability[:, None],
    aspect="auto",
    interpolation="nearest",
    cmap="viridis",
    norm=Normalize(0, 1),
)
side_axes[2].imshow(
    annotated.astype(float)[:, None],
    aspect="auto",
    interpolation="nearest",
    cmap=matplotlib.colors.ListedColormap(["#d7d7d7", "#d58a35"]),
    norm=Normalize(0, 1),
)
for side_ax, label in zip(
    side_axes,
    (
        f"Genes\n{int(np.nanmin(sizes))}–{int(np.nanmax(sizes))}",
        "Seed\nJaccard",
        "Reactome\nORA label",
    ),
):
    side_ax.set_xticks([0], [label], fontsize=7.4)
    side_ax.tick_params(
        axis="x", top=True, bottom=False, labeltop=True, labelbottom=False
    )
    side_ax.set_yticks([])
    side_ax.grid(False)
    for spine in side_ax.spines.values():
        spine.set_visible(False)

cbar = fig.colorbar(
    image,
    ax=[ax, *side_axes],
    orientation="horizontal",
    fraction=0.032,
    pad=0.055,
    aspect=48,
)
cbar.set_label(
    "Raw gene-program score difference "
    "(red = increased; blue = decreased)"
)
fig.legend(
    handles=[
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="none",
            markerfacecolor="none",
            markeredgecolor="#171717",
            markersize=5,
            label="Within-run program-effect FDR < 0.05",
        ),
        Line2D(
            [0],
            [0],
            marker="s",
            linestyle="none",
            markerfacecolor="#d58a35",
            markeredgecolor="#d58a35",
            markersize=6,
            label="Provisional Reactome within-program ORA label",
        ),
    ],
    loc="lower center",
    frameon=False,
    bbox_to_anchor=(0.5, 0.008),
    ncol=2,
)
fig.suptitle(
    "Fixed program responses with Reactome-only annotation\n"
    "row ordering is display-only and does not alter Louvain membership",
    y=0.985,
)
save_figure(
    fig, OUTPUT_DIR / "reactome_all_program_response_atlas", dpi=300
)

## 6. Reactome-only preranked GSEA

Four independent GSEA runs are performed: observed and model
conditional rankings at 24 h and 1 week. Each run tests all 319
eligible Reactome pathways with 500 gene-set permutations, a
weighting exponent of 1, size limits of 10--500 genes and seed 42.

The resulting FDR is GSEA's permutation-based pooled-null FDR
estimated within that Reactome-only run. It is not the
within-program ORA FDR and is not the program-effect FDR displayed
as circles in the response atlas.

In [ ]:
gsea_frames = []
for condition in ("IR_24h", "IR_1week"):
    condition_frame = gene_ranks[gene_ranks["condition"].eq(condition)]
    for representation, value_column in (
        ("observed", "observed_delta"),
        ("predicted_counterfactual", "predicted_counterfactual_delta"),
    ):
        rank = (
            condition_frame[["human_symbol", value_column]]
            .dropna()
            .sort_values(value_column, ascending=False)
        )
        rank.columns = ["gene", "score"]
        print(f"Running Reactome GSEA: {condition}, {representation}")
        prerank = gp.prerank(
            rnk=rank,
            gene_sets=reactome_sets,
            min_size=MIN_GENE_SET_SIZE,
            max_size=MAX_GENE_SET_SIZE,
            permutation_num=GSEA_PERMUTATIONS,
            weight=1.0,
            outdir=None,
            seed=RANDOM_SEED,
            threads=1,
            verbose=False,
        )
        result = prerank.res2d.copy().reset_index(drop=True)
        if "Term" not in result.columns:
            result = result.rename(columns={result.columns[0]: "Term"})
        result["condition"] = condition
        result["representation"] = representation
        gsea_frames.append(result)

reactome_gsea = pd.concat(gsea_frames, ignore_index=True)
for column in ["ES", "NES", "NOM p-val", "FDR q-val", "FWER p-val"]:
    reactome_gsea[column] = pd.to_numeric(
        reactome_gsea[column], errors="coerce"
    )
reactome_gsea["database"] = "REACTOME"
reactome_gsea["pathway"] = (
    reactome_gsea["Term"].astype(str).str.split("::", n=1).str[-1]
)

expected_rows = 4 * len(reactome_sets)
if len(reactome_gsea) != expected_rows:
    raise AssertionError(
        f"Expected {expected_rows:,} Reactome GSEA rows, found {len(reactome_gsea):,}"
    )
if set(reactome_gsea["Term"]) != set(reactome_sets):
    raise AssertionError("Reactome GSEA terms do not match the eligible library.")
reactome_gsea.to_csv(OUTPUT_DIR / "reactome_gsea_all.csv", index=False)
print(f"Saved {len(reactome_gsea):,} Reactome-only GSEA rows.")

## 7. Observed--model recovery across all Reactome pathways

In [ ]:
recovery_frames = []
recovery_metric_rows = []
for condition in ("IR_24h", "IR_1week"):
    observed = reactome_gsea[
        reactome_gsea["condition"].eq(condition)
        & reactome_gsea["representation"].eq("observed")
    ]
    predicted = reactome_gsea[
        reactome_gsea["condition"].eq(condition)
        & reactome_gsea["representation"].eq(
            "predicted_counterfactual"
        )
    ]
    merged = observed[
        ["Term", "database", "pathway", "NES", "FDR q-val"]
    ].merge(
        predicted[["Term", "NES", "FDR q-val"]],
        on="Term",
        suffixes=("_observed", "_predicted"),
        validate="one_to_one",
    )
    merged["condition"] = condition
    merged["sign_concordant"] = (
        np.sign(merged["NES_observed"])
        == np.sign(merged["NES_predicted"])
    )
    merged["absolute_observed_NES"] = merged["NES_observed"].abs()
    recovery_frames.append(merged)

    pearson_r, pearson_p = safe_pearson(
        merged["NES_observed"], merged["NES_predicted"]
    )
    spearman_r, spearman_p = safe_spearman(
        merged["NES_observed"], merged["NES_predicted"]
    )
    recovery_metric_rows.append(
        {
            "condition": condition,
            "n_reactome_pathways": len(merged),
            "pearson_nes": pearson_r,
            "pearson_p": pearson_p,
            "spearman_nes": spearman_r,
            "spearman_p": spearman_p,
            "directional_concordance": float(
                merged["sign_concordant"].mean()
            ),
            "n_observed_gsea_q_lt_0_05": int(
                merged["FDR q-val_observed"].lt(0.05).sum()
            ),
            "n_model_gsea_q_lt_0_05": int(
                merged["FDR q-val_predicted"].lt(0.05).sum()
            ),
        }
    )

reactome_recovery = pd.concat(recovery_frames, ignore_index=True)
recovery_metrics = pd.DataFrame(recovery_metric_rows)
reactome_recovery.to_csv(
    OUTPUT_DIR / "reactome_pathway_observed_vs_model.csv", index=False
)
recovery_metrics.to_csv(
    OUTPUT_DIR / "reactome_pathway_recovery_metrics.csv", index=False
)
display(recovery_metrics)

fig, axes = plt.subplots(1, 2, figsize=(14.2, 6.5), sharex=True, sharey=True)
all_limits = np.nanmax(
    np.abs(
        reactome_recovery[
            ["NES_observed", "NES_predicted"]
        ].to_numpy(dtype=float)
    )
)
all_limits = float(all_limits) * 1.08
for ax, condition, title in zip(
    axes,
    ("IR_24h", "IR_1week"),
    ("24 h minus control", "1 week minus control"),
):
    frame = reactome_recovery[
        reactome_recovery["condition"].eq(condition)
    ].copy()
    observed_significant = frame["FDR q-val_observed"].lt(0.05)
    model_significant = frame["FDR q-val_predicted"].lt(0.05)
    colours = np.where(
        observed_significant,
        -np.log10(
            frame["FDR q-val_observed"].clip(
                lower=np.nextafter(0, 1)
            )
        ),
        0,
    )
    scatter = ax.scatter(
        frame["NES_observed"],
        frame["NES_predicted"],
        c=colours,
        cmap="magma",
        vmin=0,
        vmax=max(2.0, float(np.nanmax(colours))),
        s=28,
        alpha=0.72,
        edgecolor="none",
    )
    ax.scatter(
        frame.loc[model_significant, "NES_observed"],
        frame.loc[model_significant, "NES_predicted"],
        s=48,
        facecolor="none",
        edgecolor="#168aad",
        linewidth=0.9,
    )
    ax.plot(
        [-all_limits, all_limits],
        [-all_limits, all_limits],
        linestyle="--",
        color="#3c3c3c",
        linewidth=1,
    )
    ax.axhline(0, color="#8c8c8c", linewidth=0.7)
    ax.axvline(0, color="#8c8c8c", linewidth=0.7)
    metric = recovery_metrics[
        recovery_metrics["condition"].eq(condition)
    ].iloc[0]
    ax.set_title(
        f"{title}\n"
        f"Pearson r={metric['pearson_nes']:.3f}; "
        f"Spearman ρ={metric['spearman_nes']:.3f}; "
        f"same direction={metric['directional_concordance']:.1%}"
    )
    ax.set_xlim(-all_limits, all_limits)
    ax.set_ylim(-all_limits, all_limits)
    ax.set_xlabel("Observed Reactome NES")
axes[0].set_ylabel("Model Reactome NES")
cbar = fig.colorbar(
    scatter,
    ax=axes,
    orientation="vertical",
    pad=0.025,
    fraction=0.035,
)
cbar.set_label(
    r"Observed significance: $-\log_{10}(q)$ "
    "(zero denotes observed q ≥ 0.05)"
)
fig.legend(
    handles=[
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="none",
            markerfacecolor="none",
            markeredgecolor="#168aad",
            markersize=7,
            label="Model Reactome GSEA q < 0.05",
        ),
        Line2D(
            [0],
            [0],
            linestyle="--",
            color="#3c3c3c",
            label="Observed = model",
        ),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.015),
    ncol=2,
    frameon=False,
)
fig.suptitle(
    "Observed versus model recovery across all 319 Reactome pathways",
    y=0.99,
)
fig.subplots_adjust(bottom=0.17, top=0.84, left=0.08, right=0.90, wspace=0.13)
save_figure(fig, OUTPUT_DIR / "reactome_all_pathway_recovery_scatter")

## 8. Representative Reactome pathways

Ten pathways per timepoint are selected using observed results
only: observed GSEA FDR, then absolute observed NES. For
presentation, highly overlapping Reactome sets are greedily
suppressed at a pathway-gene Jaccard threshold of 0.50. This
redundancy filter changes only the displayed labels; all 319
pathways remain in the inferential analysis.

In [ ]:
selected_by_condition = {}
selected_frames = []
for condition in ("IR_24h", "IR_1week"):
    condition_recovery = reactome_recovery[
        reactome_recovery["condition"].eq(condition)
    ]
    selected = select_representative_pathways(
        condition_recovery,
        reactome_sets,
        n=N_PATHWAYS_PER_TIMEPOINT,
        jaccard_threshold=REDUNDANCY_JACCARD_THRESHOLD,
    )
    selected["condition"] = condition
    selected["display_order"] = np.arange(1, len(selected) + 1)
    selected_by_condition[condition] = selected
    selected_frames.append(selected)

selected_pathways = pd.concat(selected_frames, ignore_index=True)
selected_pathways.to_csv(
    OUTPUT_DIR / "reactome_top_pathways_figure_data.csv", index=False
)

fig, axes = plt.subplots(1, 2, figsize=(19.0, 10.5))
for ax, condition, title in zip(
    axes,
    ("IR_24h", "IR_1week"),
    ("24-hour Reactome pathways", "1-week Reactome pathways"),
):
    frame = selected_by_condition[condition].sort_values("NES_observed")
    y = np.arange(len(frame))
    ax.hlines(
        y,
        frame["NES_observed"],
        frame["NES_predicted"],
        color="#b5b5b5",
        linewidth=1.4,
    )
    for values, fdr, marker, colour in (
        (
            frame["NES_observed"],
            frame["FDR q-val_observed"],
            "o",
            "#3b6ea8",
        ),
        (
            frame["NES_predicted"],
            frame["FDR q-val_predicted"],
            "s",
            "#d58a35",
        ),
    ):
        significant = fdr < 0.05
        ax.scatter(
            values[significant],
            y[significant],
            s=68,
            marker=marker,
            facecolors=colour,
            edgecolors=colour,
            zorder=3,
        )
        ax.scatter(
            values[~significant],
            y[~significant],
            s=68,
            marker=marker,
            facecolors="none",
            edgecolors=colour,
            linewidth=1.2,
            zorder=3,
        )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_yticks(
        y,
        [readable_reactome_name(term, width=38) for term in frame["Term"]],
    )
    ax.set_xlabel("Normalised enrichment score (NES)")
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.2)
    ax.grid(axis="y", visible=False)

fig.legend(
    handles=[
        Line2D(
            [],
            [],
            linestyle="none",
            marker="o",
            markersize=8,
            markerfacecolor="#3b6ea8",
            markeredgecolor="#3b6ea8",
            label="Observed",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker="s",
            markersize=8,
            markerfacecolor="#d58a35",
            markeredgecolor="#d58a35",
            label="Model conditional contrast",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker="o",
            markersize=8,
            markerfacecolor="#666666",
            markeredgecolor="#666666",
            label="Reactome GSEA q < 0.05 (filled)",
        ),
        Line2D(
            [],
            [],
            linestyle="none",
            marker="o",
            markersize=8,
            markerfacecolor="none",
            markeredgecolor="#666666",
            label="Reactome GSEA q ≥ 0.05 (hollow)",
        ),
    ],
    frameon=False,
    loc="lower center",
    ncol=4,
    bbox_to_anchor=(0.5, 0.005),
)
fig.suptitle(
    "Observed versus model-predicted Reactome pathway direction",
    y=0.985,
)
fig.tight_layout(rect=(0, 0.10, 1, 0.94), w_pad=4)
save_figure(fig, OUTPUT_DIR / "reactome_pathway_observed_vs_model")

display(
    selected_pathways[
        [
            "condition",
            "pathway",
            "NES_observed",
            "FDR q-val_observed",
            "NES_predicted",
            "FDR q-val_predicted",
            "sign_concordant",
        ]
    ]
)

## 9. Leading-edge genes for the displayed Reactome pathways

In [ ]:
leading_rows = []
for condition, selected in selected_by_condition.items():
    observed_result = reactome_gsea[
        reactome_gsea["condition"].eq(condition)
        & reactome_gsea["representation"].eq("observed")
        & reactome_gsea["Term"].isin(selected["Term"])
    ]
    for _, row in observed_result.iterrows():
        lead_genes = [
            gene
            for gene in str(row["Lead_genes"]).split(";")
            if gene and gene != "nan"
        ]
        for order, gene in enumerate(lead_genes, start=1):
            leading_rows.append(
                {
                    "condition": condition,
                    "Term": row["Term"],
                    "pathway": row["pathway"],
                    "leading_edge_order": order,
                    "human_symbol": gene,
                    "observed_NES": row["NES"],
                    "observed_GSEA_q": row["FDR q-val"],
                }
            )

reactome_leading_edges = pd.DataFrame(leading_rows)
rank_wide = (
    gene_ranks.pivot(
        index="human_symbol",
        columns="condition",
        values=["observed_delta", "predicted_counterfactual_delta"],
    )
)
rank_wide.columns = [
    f"{condition}__{representation}"
    for representation, condition in rank_wide.columns
]
rank_wide = rank_wide.rename(
    columns={
        "IR_24h__observed_delta": "Observed 24 h",
        "IR_24h__predicted_counterfactual_delta": "Model 24 h",
        "IR_1week__observed_delta": "Observed 1 week",
        "IR_1week__predicted_counterfactual_delta": "Model 1 week",
    }
)

unique_leading = reactome_leading_edges[
    ["human_symbol", "condition"]
].drop_duplicates()
gene_priority = (
    unique_leading.groupby("human_symbol")["condition"]
    .nunique()
    .rename("n_timepoints")
    .to_frame()
    .join(rank_wide)
)
effect_columns = [
    "Observed 24 h",
    "Model 24 h",
    "Observed 1 week",
    "Model 1 week",
]
observed_columns = ["Observed 24 h", "Observed 1 week"]
gene_priority["max_observed_effect"] = gene_priority[observed_columns].max(axis=1)
gene_priority["min_observed_effect"] = gene_priority[observed_columns].min(axis=1)
gene_priority["dominant_observed_direction"] = np.where(
    gene_priority["max_observed_effect"]
    >= -gene_priority["min_observed_effect"],
    "positive",
    "negative",
)
positive_genes = (
    gene_priority[
        gene_priority["dominant_observed_direction"].eq("positive")
    ]
    .sort_values(
        ["n_timepoints", "max_observed_effect"],
        ascending=[False, False],
    )
    .head(18)
    .index.tolist()
)
negative_genes = (
    gene_priority[
        gene_priority["dominant_observed_direction"].eq("negative")
    ]
    .sort_values(
        ["n_timepoints", "min_observed_effect"],
        ascending=[False, True],
    )
    .head(18)
    .index.tolist()
)
displayed_genes = positive_genes + negative_genes

reactome_leading_edges = reactome_leading_edges.merge(
    rank_wide.reset_index(), on="human_symbol", how="left"
)
reactome_leading_edges["displayed_in_heatmap"] = (
    reactome_leading_edges["human_symbol"].isin(displayed_genes)
)
reactome_leading_edges.to_csv(
    OUTPUT_DIR / "reactome_leading_edge_genes.csv", index=False
)

heat = rank_wide.reindex(displayed_genes)[effect_columns]
limit = max(float(np.nanmax(np.abs(heat.to_numpy(dtype=float)))), 1e-9)
fig, ax = plt.subplots(
    figsize=(9.4, max(8.0, 0.29 * len(heat) + 2.7))
)
sns.heatmap(
    heat,
    cmap="RdBu_r",
    center=0,
    vmin=-limit,
    vmax=limit,
    linewidths=0.25,
    linecolor="white",
    cbar_kws={"label": "Gene-level treated-minus-control effect"},
    ax=ax,
)
ax.set_title(
    "Leading-edge genes from the displayed observed Reactome pathways\n"
    "balanced positive and negative genes selected without using model agreement"
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
save_figure(fig, OUTPUT_DIR / "reactome_leading_edge_gene_heatmap")

## 10. Audit against the mixed Hallmark + Reactome run

This table is a sensitivity audit, not a second inference. It
compares each Reactome-only GSEA result with the same pathway from
the original mixed-library run. The primary purpose is to show why
Reactome-only q-values cannot be obtained by filtering the old
table: the pooled FDR family has changed.

In [ ]:
old_reactome = combined_gsea[
    combined_gsea["database"].eq("REACTOME")
][
    [
        "Term",
        "condition",
        "representation",
        "NES",
        "NOM p-val",
        "FDR q-val",
    ]
].copy()
fdr_audit = reactome_gsea[
    [
        "Term",
        "condition",
        "representation",
        "NES",
        "NOM p-val",
        "FDR q-val",
    ]
].merge(
    old_reactome,
    on=["Term", "condition", "representation"],
    how="left",
    suffixes=("_reactome_only", "_mixed_library"),
    validate="one_to_one",
)
fdr_audit["gsea_q_change_reactome_only_minus_mixed"] = (
    fdr_audit["FDR q-val_reactome_only"]
    - fdr_audit["FDR q-val_mixed_library"]
)
fdr_audit.to_csv(
    OUTPUT_DIR / "reactome_fdr_audit_vs_mixed_library.csv", index=False
)
display(
    fdr_audit.groupby(["condition", "representation"])
    .agg(
        median_absolute_nes_difference=(
            "NES_reactome_only",
            lambda x: np.nan,
        )
    )
    .reset_index()
    .drop(columns="median_absolute_nes_difference")
    .merge(
        fdr_audit.assign(
            absolute_nes_difference=lambda x: (
                x["NES_reactome_only"] - x["NES_mixed_library"]
            ).abs(),
            absolute_q_difference=lambda x: (
                x["FDR q-val_reactome_only"]
                - x["FDR q-val_mixed_library"]
            ).abs(),
        )
        .groupby(["condition", "representation"], as_index=False)
        .agg(
            median_absolute_nes_difference=(
                "absolute_nes_difference",
                "median",
            ),
            median_absolute_q_difference=(
                "absolute_q_difference",
                "median",
            ),
            max_absolute_q_difference=(
                "absolute_q_difference",
                "max",
            ),
        ),
        on=["condition", "representation"],
        how="left",
    )
)

## 11. Reproducibility manifest and concise findings

In [ ]:
software_versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn": sns.__version__,
    "gseapy": gp.__version__,
}
manifest = {
    "analysis": "Reactome-only embedding-program ORA and ranked pathway GSEA",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis_directory": str(ANALYSIS_DIR),
    "output_directory": str(OUTPUT_DIR),
    "source_output_directory": str(SOURCE_OUTPUT_DIR),
    "upstream_results_frozen": [
        "checkpoint embeddings",
        "embedding-neighbour graph",
        "Louvain partition",
        "program scores and program-effect tests",
        "observed and model gene-effect rankings",
    ],
    "reactome_gmt": str(REACTOME_GMT),
    "reactome_gmt_sha256": sha256_file(REACTOME_GMT),
    "raw_reactome_pathways": len(raw_reactome),
    "eligible_reactome_pathways": len(reactome_sets),
    "gene_universe_size": len(universe),
    "louvain_program_count": int(membership["program_id"].nunique()),
    "ora_test_count": len(reactome_program_ora),
    "ora_within_program_fdr_family": len(reactome_sets),
    "ora_global_fdr_family": len(reactome_program_ora),
    "gsea_runs": 4,
    "gsea_permutations": GSEA_PERMUTATIONS,
    "gsea_weight": 1.0,
    "minimum_gene_set_size": MIN_GENE_SET_SIZE,
    "maximum_gene_set_size": MAX_GENE_SET_SIZE,
    "random_seed": RANDOM_SEED,
    "display_pathways_per_timepoint": N_PATHWAYS_PER_TIMEPOINT,
    "display_redundancy_jaccard_threshold": REDUNDANCY_JACCARD_THRESHOLD,
    "input_sha256": {
        name: sha256_file(path) for name, path in INPUT_FILES.items()
    },
    "software_versions": software_versions,
}
with (OUTPUT_DIR / "reactome_analysis_manifest.json").open("w") as handle:
    json.dump(manifest, handle, indent=2)

metric_lookup = recovery_metrics.set_index("condition")
summary_lines = [
    "# Reactome-only analysis summary",
    "",
    (
        f"The fixed 2,439-gene universe yielded {len(reactome_sets)} eligible "
        "Reactome pathways after the 10--500 gene filter."
    ),
    (
        f"Program ORA evaluated {len(reactome_program_ora):,} fixed "
        "program--pathway pairs. "
        f"{len(provisional_hits)} pairs across "
        f"{provisional_hits['program_id'].nunique()} programs met the "
        "provisional within-program FDR and overlap rule. "
        f"No association passed Reactome-global FDR < 0.05; the minimum "
        f"global FDR was {reactome_program_ora['reactome_global_fdr'].min():.3f}."
    ),
    "",
]
for condition, label in (
    ("IR_24h", "24 h"),
    ("IR_1week", "1 week"),
):
    row = metric_lookup.loc[condition]
    summary_lines.append(
        f"- {label}: observed--model Reactome NES Pearson "
        f"r = {row['pearson_nes']:.3f}, Spearman rho = "
        f"{row['spearman_nes']:.3f}, and directional concordance = "
        f"{row['directional_concordance']:.1%}. "
        f"Observed/model Reactome GSEA q < 0.05 counts were "
        f"{int(row['n_observed_gsea_q_lt_0_05'])}/"
        f"{int(row['n_model_gsea_q_lt_0_05'])}."
    )
summary_lines.extend(
    [
        "",
        "These pathway-recovery correlations are descriptive because Reactome "
        "pathways are nested and statistically dependent. A significant observed "
        "pathway and a non-significant model pathway do not constitute a formal "
        "test of an observed--model difference.",
        "",
        "The program-effect FDR, Reactome within-program ORA FDR, Reactome-global "
        "ORA FDR, and Reactome GSEA pathway FDR answer different questions and "
        "must not be treated as interchangeable.",
    ]
)
(OUTPUT_DIR / "reactome_analysis_summary.md").write_text(
    "\n".join(summary_lines) + "\n"
)

output_inventory = pd.DataFrame(
    [
        {
            "file": path.name,
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for path in sorted(OUTPUT_DIR.iterdir())
        if path.is_file()
    ]
)
output_inventory.to_csv(
    OUTPUT_DIR / "reactome_output_inventory.csv", index=False
)
print((OUTPUT_DIR / "reactome_analysis_summary.md").read_text())
print(f"\nCreated {len(output_inventory) + 1} files in {OUTPUT_DIR}")

## Interpretation guardrails

- Reactome pathway enrichment is independent of the Louvain
  partition; the program atlas only reconnects those effects to the
  fixed embedding-derived grouping.
- Embedding proximity is an undirected learned association, not a
  regulatory, directional, or causal edge.
- Reactome pathways are nested and overlapping, so related
  significant terms are not independent confirmations.
- The model result is a matched *in silico* conditional contrast
  obtained from control-cell inputs, not a causal or longitudinal
  counterfactual.
- The Reactome-only GSEA q-values in this notebook are the valid
  values for this pathway family. The mixed-library Reactome subset
  remains useful only as a sensitivity comparison.